In [ ]:
import neuromaps
print("neuromaps version:", neuromaps.__version__)
# Add more imports if needed

from neuromaps.datasets import fetch_atlas ## used to access the templates for the coordinate system
import nibabel as nib ## used to load system dictionary per key
from neuromaps.datasets import available_annotations ## repository of brain maps - spatial maps representing some
from neuromaps.datasets import available_tags ## most annotations have “tags” that help to describe the data they represent
from neuromaps.datasets import fetch_annotation
from neuromaps.datasets import fetch_fsaverage

from neuromaps import transforms
import netneurotools
# possibly need
from netneurotools import datasets as nntdata
from neuromaps import parcellate
from neuromaps.parcellate import Parcellater
from neuromaps.images import dlabel_to_gifti
# plotting 
from neuromaps.images import load_data
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from neuromaps import plotting
from nilearn import plotting
import numpy as np
import pandas as pd
# sampling
from neuromaps import datasets, images, nulls, resampling
from neuromaps.resampling import resample_images
from neuromaps.stats import compare_images
from neuromaps import stats
from nilearn.datasets import fetch_atlas_surf_destrieux
from neuromaps.nulls import alexander_bloch
from neuromaps.stats import compare_images
from scipy.stats import pearsonr

from nilearn.surface import load_surf_mesh
from brainspace.null_models import SpinPermutations
from nilearn.surface import InMemoryMesh, PolyMesh
from nilearn.surface import SurfaceImage
from nilearn.plotting import view_surf

import time
# for FDR
from statsmodels.stats.multitest import multipletests
from neuromaps.nulls import burt2018

In [2]:
evo_map = fetch_annotation(source='hill2010', desc='evoexp', space='fsLR', den='164k')


[References] Please cite the following papers if you are using this data:

  For {'source': 'hill2010', 'desc': 'evoexp', 'space': 'fsLR', 'den': '164k'}:
  [primary]:
    Jason Hill, Terrie Inder, Jeffrey Neil, Donna Dierker, John Harwell, and David Van Essen. Similar patterns of cortical expansion during human development and evolution. Proceedings of the National Academy of Sciences, 107(29):13135–13140, 2010.
  [secondary]:
    


In [23]:
evo_map

'/Users/kevin/neuromaps-data/annotations/hill2010/evoexp/fsLR/source-hill2010_desc-evoexp_space-fsLR_den-164k_hemi-R_feature.func.gii'

In [ ]:
genepc1 = fetch_annotation(source='abagen', desc='genepc1', space='fsaverage', den='10k')
T1w_T2w_ratio = fetch_annotation(source='hcps1200', desc='myelinmap',space='fsLR', den='32k')
intersub_var = fetch_annotation(source='mueller2013', desc='intersubjvar', space='fsLR', den='164k')
scalingnih = fetch_annotation(source='reardon2018', desc='scalingnih', space='civet', den='41k')


[References] Please cite the following papers if you are using this data:

  For {'source': 'abagen', 'desc': 'genepc1', 'space': 'fsaverage', 'den': '10k'}:
  [primary]:
    Michael J Hawrylycz, Ed S Lein, Angela L Guillozet-Bongaarts, Elaine H Shen, Lydia Ng, Jeremy A Miller, Louie N Van De Lagemaat, Kimberly A Smith, Amanda Ebbert, Zackery L Riley, and others. An anatomically comprehensive atlas of the adult human brain transcriptome. Nature, 489(7416):391, 2012.
    Ross D Markello, Aurina Arnatkeviciute, Jean-Baptiste Poline, Ben D Fulcher, Alex Fornito, and Bratislav Misic. Standardizing workflows in imaging transcriptomics with the abagen toolbox. eLife, 10:e72129, 2021.
  [secondary]:
    

[References] Please cite the following papers if you are using this data:

  For {'source': 'hcps1200', 'desc': 'myelinmap', 'space': 'fsLR', 'den': '32k'}:
  [primary]:
    Matthew F Glasser, Timothy S Coalson, Emma C Robinson, Carl D Hacker, John Harwell, Essa Yacoub, Kamil Ugurbil, Jespe

In [27]:
import numpy as np
from scipy.stats import pearsonr

from neuromaps.images import load_data
from neuromaps.transforms import fslr_to_fsaverage
from neuromaps.parcellate import Parcellater
from neuromaps.datasets import fetch_schaefer2018
from neuromaps.nulls import cornblath
from neuromaps.stats import compare_images

# ---------------------------------------------------------
# 1. Transform evo_map from fsLR to fsaverage 10k (right hemi)
# ---------------------------------------------------------

evo_map_R_path = '/Users/kevin/neuromaps-data/annotations/hill2010/evoexp/fsLR/source-hill2010_desc-evoexp_space-fsLR_den-164k_hemi-R_feature.func.gii'

# evo_map only has RH data, so pass hemi='R'
evo_in_fsaverage10k = fslr_to_fsaverage(evo_map_R_path, '10k', hemi='R')

# This should be a RH fsaverage 10k .func.gii
evo_R = load_data(evo_in_fsaverage10k[0])

genepc1_R = load_data(genepc1[1])   # right hemisphere fsaverage 10k

assert evo_R.shape == genepc1_R.shape, (
    f"RH evo and gene maps must have same length; "
    f"got evo_R {evo_R.shape}, genepc1_R {genepc1_R.shape}"
)

# ---------------------------------------------------------
# 3. Build full L+R vertex maps (NaNs for missing left hemi)
# ---------------------------------------------------------

nan_L = np.full_like(evo_R, np.nan)

# Full surface: [L vertices, R vertices]
evo_full = np.concatenate([nan_L, evo_R])
genepc1_full = np.concatenate([np.full_like(genepc1_R, np.nan), genepc1_R])

print("evo_full shape:", evo_full.shape)
print("genepc1_full shape:", genepc1_full.shape)

# ---------------------------------------------------------
# 4. Vertex-wise empirical correlation (sanity check)
# ---------------------------------------------------------

mask_vertex = ~np.isnan(evo_full) & ~np.isnan(genepc1_full)
r_emp_vertex, _ = pearsonr(evo_full[mask_vertex], genepc1_full[mask_vertex])
print(f"Empirical vertex-wise correlation = {r_emp_vertex:.3f}")

# ---------------------------------------------------------
# 5. Get a Schaefer2018 fsaverage parcellation via neuromaps
# ---------------------------------------------------------

# This is the key change: use fetch_schaefer2018, not fetch_atlas
schaefer = fetch_schaefer2018(space='fsaverage', density='10k')

# The returned dict has keys like '100Parcels7Networks', '200Parcels7Networks', etc.
# We'll pick 200 parcels, 7 networks:
schaefer200_7 = schaefer['200Parcels7Networks']

L_PARC_PATH = schaefer200_7['L']   # lh parcellation func.gii
R_PARC_PATH = schaefer200_7['R']   # rh parcellation func.gii

print("Left parcellation path:", L_PARC_PATH)
print("Right parcellation path:", R_PARC_PATH)

# ---------------------------------------------------------
# 6. Create Parcellater and parcellate both maps
# ---------------------------------------------------------

parc = Parcellater(
    parcellation=(L_PARC_PATH, R_PARC_PATH),
    space='fsaverage',
    density='10k'
)

# evo_full & genepc1_full are vertex-wise [L+R] arrays in fsaverage 10k
evo_parc = parc.fit_transform(evo_full, space='fsaverage').squeeze()
genepc1_parc = parc.fit_transform(genepc1_full, space='fsaverage').squeeze()

print("evo_parc shape:", evo_parc.shape)
print("genepc1_parc shape:", genepc1_parc.shape)

# ---------------------------------------------------------
# 7. Empirical parcel-wise correlation (RH-only effectively)
# ---------------------------------------------------------

mask_parc = ~np.isnan(evo_parc) & ~np.isnan(genepc1_parc)
r_emp_parc, _ = pearsonr(evo_parc[mask_parc], genepc1_parc[mask_parc])
print(f"Empirical parcel-wise correlation = {r_emp_parc:.3f}")

# ---------------------------------------------------------
# 8. Generate Cornblath nulls (parcellated data)
# ---------------------------------------------------------

nulls = cornblath(
    evo_parc,
    atlas='fsaverage',
    density='10k',
    parcellation=(L_PARC_PATH, R_PARC_PATH),
    n_perm=1000
)

print("nulls shape:", nulls.shape)

# ---------------------------------------------------------
# 9. Compare evo vs gene with spin-based nulls (parcel space)
# ---------------------------------------------------------

r_emp_gene, p_spin_gene, nulls_gene = compare_images(
    evo_parc,
    genepc1_parc,
    nulls=nnulls,
    metric='pearsonr',
    return_nulls=True
)

print(f"Spatial correlation (parcel-wise) = {r_emp_gene:.3f}, "
      f"spin test p = {p_spin_gene:.4f}")

ImportError: cannot import name 'fetch_schaefer2018' from 'neuromaps.datasets' (/opt/anaconda3/envs/neuromaps/lib/python3.11/site-packages/neuromaps/datasets/__init__.py)

In [28]:
from neuromaps.datasets.atlases import DENSITIES
print(DENSITIES.keys())

dict_keys(['civet', 'fsaverage', 'fsLR', 'MNI152'])


In [ ]:
### TEST TO USE FSLR 32K FOR EVO MAP TO MATCH MYELIN MAP
## it Works

# transform source map to fsaverage 10k
evo_in_fsLR = transforms.fslr_to_fslr(evo_map, '32k', hemi='R') ## (<nibabel.gifti.gifti.GiftiImage at 0x3156bff10>,)
# load the data --> bascially reads in the .func.gii file and returns the data array
evo_data = load_data(evo_in_fsLR[0]) # load_data([path where string of the data file])
T1w_T2w_ratio_data = load_data(T1w_T2w_ratio) 

# make left hemi NaN's so spin test works for built in function
evo_full = np.concatenate([np.full_like(evo_data, np.nan), evo_data])
myelin_full = T1w_T2w_ratio_data
print(f"evo_fill shape: {evo_full.shape}")

# empirical correlation
mask = ~np.isnan(evo_full) & ~np.isnan(myelin_full)
r_emp_prelim, _ = pearsonr(evo_full[mask], myelin_full[mask])
print(f"Empirical correlation = {r_emp_prelim:.3f}")

# spin test
nulls = burt2018(
    evo_full,
    atlas='fsLR',
    density='32k',
    n_perm=1000
)

# empirical r to null
# compare_images() compares the two maps and the nulls --> outputs r, p value, and null distribution
r_emp, p_spin, nulls_dist = compare_images(
    evo_full,
    myelin_full,
    nulls=nulls,
    metric='pearsonr',
    return_nulls=True # null distribution for boxplot later
)
print(f"Spatial correlation for Gene Expression = {r_emp:.3f}, spin test p = {p_spin:.4f}")


evo_fill shape: (64984,)
Empirical correlation = -0.040


KeyError: 'pial'

In [13]:
## Intersubject Variability

# both are in the same dimension already --> no need to transform
evo_data_r = load_data(evo_map)
ISV_r = load_data(intersub_var[0]) # right hemi

# make left hemi NaN's so spin test works
evo_full = np.concatenate([np.full_like(evo_data_r, np.nan), evo_data_r])
ISV_full = np.concatenate([np.full_like(ISV_r, np.nan), ISV_r])

# empirical correlation
mask = ~np.isnan(evo_full) & ~np.isnan(ISV_full)
r_emp, _ = pearsonr(evo_full[mask], ISV_full[mask])
print(f"Empirical correlation = {r_emp:.3f}")

# spin test
nulls = burt2018(
    evo_full,
    atlas='fsLR',
    density='164k',
    n_perm=1000
)

# empirical r to null
r_emp_ISV, p_spin_ISV, nulls_ISV = compare_images(
    evo_full,
    ISV_full,
    nulls=nulls,
    metric='pearsonr',
    return_nulls=True
)
print(f"Spatial correlation for Intersubject Variability = {r_emp_ISV:.3f}, spin test p = {p_spin_ISV:.4f}")

ValueError: operands could not be broadcast together with shapes (327684,) (81924,) 

In [17]:
# test against intersubject and myelin

### TEST TO USE FSLR 32K FOR EVO MAP TO MATCH MYELIN MAP
## it Works

# transform source map to fsaverage 10k
evo_in_fsLR = transforms.fslr_to_fslr(intersub_var, '32k', hemi='R') ## (<nibabel.gifti.gifti.GiftiImage at 0x3156bff10>,)
# load the data --> bascially reads in the .func.gii file and returns the data array
evo_data = load_data(evo_in_fsLR[0]) # load_data([path where string of the data file])
T1w_T2w_ratio_data = load_data(T1w_T2w_ratio) 

# make left hemi NaN's so spin test works for built in function
evo_full = np.concatenate([np.full_like(evo_data, np.nan), evo_data])
myelin_full = T1w_T2w_ratio_data
print(f"evo_fill shape: {evo_full.shape}")

# empirical correlation
mask = ~np.isnan(evo_full) & ~np.isnan(myelin_full)
r_emp_prelim, _ = pearsonr(evo_full[mask], myelin_full[mask])
print(f"Empirical correlation = {r_emp_prelim:.3f}")

# spin test
nulls = burt2018(
    evo_full,
    atlas='fslr',
    density='32k',
    n_perm=1000
)

# empirical r to null
# compare_images() compares the two maps and the nulls --> outputs r, p value, and null distribution
r_emp, p_spin, nulls_dist = compare_images(
    evo_full,
    myelin_full,
    nulls=nulls,
    metric='pearsonr',
    return_nulls=True # null distribution for boxplot later
)
print(f"Spatial correlation for Gene Expression = {r_emp:.3f}, spin test p = {p_spin:.4f}")


ValueError: Invalid density for fsLR space: 41k